<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
# 3 · Sequence Models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/03-sequence-models.ipynb)

**From Traditional NLP to Modern LLMs** · Day 1 · 95 minutes (45 briefing, 50 lab)

RNNs, LSTMs and neural language modeling, and why gradients vanish over long sequences.

**By the end of this notebook you can:**

- Implement the RNN and LSTM cell updates and the language-model loss
- Explain vanishing gradients and how gating addresses them, from measured gradient decay
- Evaluate the LSTM against the n-gram baseline in a like-for-like perplexity comparison

**Stack:** PyTorch

In [ ]:
#@title Workshop harness: run this cell first { display-mode: "form" }
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Tick WORKED_EXAMPLE to run every reference solution in place of your code. That gives a
# complete worked example; it does not show that you did the exercises.
WORKED_EXAMPLE = False  #@param {type:"boolean"}

import os as _os
import time as _time

from IPython import get_ipython as _get_ipython

_NOTEBOOK = "03-sequence-models"
_CONTENT_SHA = "ca6f60cc032b95da"
_EXERCISES = {1: ('rnn_cell_step',), 2: ('lm_loss_and_ppl',), 3: ('clip_gradients',), 4: ('lstm_cell_step',), 5: ('sample_with_temperature',), 6: ('top_k_filter', 'nucleus_filter')}
_HINTS = {2, 3, 5}  # exercises with a folded Hint cell above their TODO cell
_PATHS = {'offline': 'offline stand-ins and test doubles: checks that the code runs, not what a model does', 'open': "the notebook's own code and models, no API keys", 'keyed': 'commercial APIs with keys'}
_FALLBACK_NOTE = None
_FALLBACK_CPU_ONLY = False  # the note describes the run a CPU runtime switches to
_RECORDED_SETTINGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE', 'NLP_LLMS_QUICK', 'HF_HUB_OFFLINE')
_OFFLINE_FLAGS = ('NLP_LLMS_OFFLINE_TINY', 'NLP_LLMS_STUB', 'NLP_LLMS_LAB07_OFFLINE', 'NLP_LLMS_LAB09_OFFLINE', 'NLP_LLMS_LAB10_OFFLINE', 'NLP_LLMS_LAB13_OFFLINE', 'NLP_LLMS_LAB14_OFFLINE', 'NLP_LLMS_LAB15_OFFLINE')
_FALLBACK_MARKERS = ('Could not load ', 'did not load', 'USING StubProvider', 'LSA stand-in fitted', 'OFFLINE TEST MODE')
_RECORD_PACKAGES = (
    "torch", "transformers", "numpy", "scikit-learn", "peft", "datasets", "openai",
    "anthropic", "typesafe-sdk", "langgraph", "langchain-core", "llama-index-core",
)


class _Workshop:
    """Keeps the folded solutions from replacing your code, and says whose code each
    checkpoint checked: yours, the reference solution, or code the lab provides."""

    _MISSING = object()

    def __init__(self, old=None):
        # Rerunning this cell (as Run all does) keeps the stored solutions but starts a new
        # run: the checkpoint results, cell times and run clock describe this run only.
        self.ref = getattr(old, "ref", {})
        self.stub = getattr(old, "stub", {})
        self.chosen = getattr(old, "chosen", set())  # exercises switched by use_reference
        self.verified = {}
        self._verifying = None
        self.results = {}
        self.cell_seconds = []
        self.cell_errors = 0
        self.fallbacks = set()
        self.started = _time.time()
        self._current = None
        self._t0 = None

    @property
    def worked(self):
        if _os.environ.get("NLP_LLMS_WORKED") == "1":
            return True
        return bool(_get_ipython().user_ns.get("WORKED_EXAMPLE"))

    def _is_reference(self, n, name, obj):
        return any(obj is r for r in self.ref.get(n, {}).get(name, ()))

    def solution(self, n):
        """Decorator on a reference solution: stores it and, unless WORKED_EXAMPLE is
        ticked, leaves your own definition of the same name in place."""

        def bind(obj, name=None):
            name = name or obj.__name__
            self.ref.setdefault(n, {}).setdefault(name, []).append(obj)
            current = _get_ipython().user_ns.get(name, self._MISSING)
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.stub.setdefault(n, {})[name] = current
            if self.worked:
                return obj
            if current is not self._MISSING and not self._is_reference(n, name, current):
                self.chosen.discard(n)  # your TODO cell ran again: back to your code
                return current
            if n in self.chosen:
                return obj  # you chose the reference with use_reference(n)
            # Your TODO cell has not run, or the name still holds a reference from a
            # worked run: go back to your code if it is known, never silently to the reference.
            yours = self.stub.get(n, {}).get(name, self._MISSING)
            if yours is not self._MISSING:
                return yours
            if callable(obj):
                return self._not_run(n, name)
            print(f"[workshop] Run your # TODO {n} cell: {name} is the reference until you do.")
            return obj

        return bind

    def _not_run(self, n, name):
        """Stands in for a function or class whose TODO cell has not run yet."""

        def missing(*args, **kwargs):
            raise NotImplementedError(
                f"{name}: run your # TODO {n} cell first, or go on with"
                f" workshop.use_reference({n!r})"
            )

        missing.__name__ = name
        return missing

    def solution_value(self, n, name, value):
        """The same as solution(n), for a value rather than a function or class."""
        if value is None or isinstance(value, (bool, int, float, str, bytes)):
            # Python shares equal small values, so yours and the reference could not be told
            # apart: give such a value in the stub instead of making it part of the exercise.
            kind = type(value).__name__
            raise TypeError(f"solution_value({n!r}, {name!r}): use an object, not a plain {kind}")
        return self.solution(n)(value, name)

    def use_reference(self, n):
        """Go on with the reference solution of exercise n in place of yours."""
        if n not in self.ref:
            raise KeyError(f"Exercise {n} has no reference solution yet: run its Solution cell")
        ns = _get_ipython().user_ns
        for name, objs in self.ref[n].items():
            ns[name] = objs[-1]
        self.chosen.add(n)
        print(
            f"Exercise {n}: now using the REFERENCE solution. Rerun the cells below it."
            f" To switch back to your code, rerun your # TODO {n} cell."
        )

    def checkpoint(self, n=None, label=None):
        """First line of a checkpoint cell: notes whose code the cell is about to check."""
        names = _EXERCISES.get(n, ())
        ns = _get_ipython().user_ns
        k = sum(self._is_reference(n, x, ns.get(x, self._MISSING)) for x in names)
        if not names:
            whose = None  # the results of the cells above: provided code, maybe your earlier work
        elif k == len(names):
            whose = "the REFERENCE solution"
        elif k == 0:
            whose = "your code"
        else:
            whose = "partly the REFERENCE solution"
        self._current = (n, str(label if label is not None else n), whose)

    def _pre(self, info):
        self._t0 = _time.time()

    def _post(self, result):
        if self._t0 is None:  # this harness cell itself: its hooks were registered mid-cell
            return
        self.cell_seconds.append(round(_time.time() - self._t0, 2))
        if not result.success and self._verifying is None:
            self.cell_errors += 1  # a verification copy fails on purpose; it is not an error
        if self._current is None:
            return
        n, label, whose = self._current
        self._current = None
        if self._verifying is not None:
            self.verified[self._verifying] = bool(result.success)
            return
        self.results[label] = (bool(result.success), whose)
        if whose is None:
            if result.success:
                print(f"[workshop] Checkpoint {label} passed.")
            elif isinstance(result.error_in_exec, NotImplementedError):
                print(
                    f"[workshop] Checkpoint {label} uses an exercise you have not written yet."
                    " Finish it, or go on with workshop.use_reference(N) for that exercise."
                )
            else:
                print(f"[workshop] Checkpoint {label} failed. Read the message above.")
        elif result.success:
            print(f"[workshop] Checkpoint {label} passed on {whose}.")
        elif isinstance(result.error_in_exec, NotImplementedError):
            print(
                f"[workshop] Checkpoint {label}: TODO {n} is not written yet. Write it, run"
                " its cell, then rerun this one. To go on without it:"
                f" workshop.use_reference({n!r})"
            )
            if n in _HINTS:
                print(f"[workshop] Stuck? Open the folded Hint above TODO {n} first.")
        elif whose == "the REFERENCE solution":
            print(
                f"[workshop] Checkpoint {label} failed on the reference solution: a problem"
                " with the lab or the runtime, not with your code. Tell the instructor."
            )
        else:
            print(
                f"[workshop] Checkpoint {label} failed on {whose}. Read the assertion above,"
                f" fix TODO {n}, or go on with workshop.use_reference({n!r})."
            )
            if n in _HINTS:
                print(f"[workshop] The folded Hint above TODO {n} says what to check.")

    def _verify_begin(self, n):
        """Test-only (scripts/test_notebooks.py --verify-checkpoints): bind exercise n's
        stubs, so that a copy of its checkpoint runs on unfinished code."""
        ns = _get_ipython().user_ns
        self._saved = {name: ns.get(name, self._MISSING) for name in _EXERCISES.get(n, ())}
        ns.update(self.stub.get(n, {}))
        self._verifying = n

    def _verify_end(self, n):
        """Test-only: restore the reference and require that the stub copy failed."""
        ns = _get_ipython().user_ns
        for name, value in self._saved.items():
            if value is self._MISSING:
                ns.pop(name, None)
            else:
                ns[name] = value
        self._verifying = None
        if self.verified.get(n) is not False:
            raise AssertionError(
                f"The checkpoint of exercise {n} passed on the unfinished stub: it cannot tell"
                " a finished exercise from an unfinished one"
            )

    def summary(self):
        """Which checkpoints passed, and on whose code."""
        groups = {"your code": [], "the reference": [], "other checks": [], "failed": []}
        for label, (ok, whose) in self.results.items():
            if not ok:
                groups["failed"].append(label)
            elif whose is None:
                groups["other checks"].append(label)
            elif whose == "your code":
                groups["your code"].append(label)
            else:
                groups["the reference"].append(label)
        mode = "WORKED EXAMPLE: reference solutions" if self.worked else "your code"
        print(f"{_NOTEBOOK}, run with {mode}.")
        path = self.path()
        # _PATHS comes from readiness.paths in _variables.yml; the offline entry says that
        # such numbers check the code, not a model.
        print(f"  What ran: {_PATHS[path]}.")
        # The labs choose their shorter CPU run from DEVICE, and Labs 7 and 10 let a participant
        # force the full run with FAST = False: a CPU-only note describes only the shorter run.
        ns = _get_ipython().user_ns
        short = ns["FAST"] if "FAST" in ns else not str(ns.get("DEVICE", "")).startswith("cuda")
        if path == "open" and _FALLBACK_NOTE and (short or not _FALLBACK_CPU_ONLY):
            print(f"  Read the numbers with this in mind: {_FALLBACK_NOTE}.")
        lines = {
            "your code": "Checkpoints passed on your code",
            "the reference": "Checkpoints passed on a reference solution",
            "other checks": "Other checks passed",
            "failed": "Checkpoints failed",
        }
        for group, labels in groups.items():
            print(f"  {lines[group]}: {', '.join(labels) or 'none'}")
        if self.worked:
            print("  A worked-example run shows how the lab goes, not that you did it.")

    def path(self):
        """offline, keyed or open: the same rule as scripts/add_run_record.py's path_of."""
        ns = _get_ipython().user_ns
        if self.fallbacks or any(_os.environ.get(flag) for flag in _OFFLINE_FLAGS):
            return "offline"
        if ns.get("PROVIDER") in ("openai", "anthropic") or ns.get("JEV_PATH") == "keyed":
            return "keyed"
        return "open"

    def run_record(self):
        """A run record for runs/ (see runs/README.md): paste it into
        scripts/add_run_record.py. It holds no code, outputs, keys or names."""
        import importlib.metadata as _md
        import json
        import platform
        import sys

        gpu = None
        try:
            import torch

            if torch.cuda.is_available():
                gpu = torch.cuda.get_device_name(0)
        except Exception:
            pass
        packages = {}
        for name in _RECORD_PACKAGES:
            try:
                packages[name] = _md.version(name)
            except Exception:
                pass
        import datetime as _dt

        record = {
            "date": _dt.datetime.now(_dt.timezone.utc).date().isoformat(),
            "notebook": _NOTEBOOK,
            "content_sha": _CONTENT_SHA,
            "mode": "worked" if self.worked else "learner",
            # A run passes when it reached this last cell with no error on the way and every
            # checkpoint passed. Run all reruns the harness first, so it starts clean.
            "status": "pass"
            if self.results and self.cell_errors == 0 and all(ok for ok, _ in self.results.values())
            else "fail",
            "cell_errors": self.cell_errors,
            "fallbacks": sorted(self.fallbacks),
            "seconds": round(_time.time() - self.started, 1),
            "cell_seconds": self.cell_seconds,
            "python": platform.python_version(),
            "platform": sys.platform,
            "gpu": gpu,
            "colab_release": _os.environ.get("COLAB_RELEASE_TAG"),
            "provider": _get_ipython().user_ns.get("PROVIDER"),
            "jev_path": _get_ipython().user_ns.get("JEV_PATH"),
            "settings": {k: _os.environ[k] for k in _RECORDED_SETTINGS if _os.environ.get(k)},
            "packages": packages,
        }
        print("----- run record (paste into scripts/add_run_record.py) -----")
        print(json.dumps(record))
        print("----- end of run record -----")


_ip = _get_ipython()
_old = _ip.user_ns.get("workshop")
# Remove only an earlier harness's own hooks; every other hook (Colab's, an extension's)
# stays registered.
for _event in ("pre_run_cell", "post_run_cell"):
    for _callback in list(_ip.events.callbacks[_event]):
        if type(getattr(_callback, "__self__", None)).__name__ == "_Workshop":
            _ip.events.unregister(_event, _callback)
workshop = _Workshop(_old if hasattr(_old, "ref") else None)

# Notice when a lab falls back to a stand-in, for the run record. print is wrapped once.
import builtins as _builtins

if not getattr(_builtins.print, "_workshop", False):
    _print = _builtins.print

    def _watching_print(*args, **kwargs):
        try:
            ip = _get_ipython()
            ws = ip.user_ns.get("workshop") if ip is not None else None
            if ws is not None and hasattr(ws, "fallbacks"):
                # Labs print their fallback notices as strings; never stringify other objects.
                text = " ".join(a for a in args if isinstance(a, str))
                ws.fallbacks.update(m.strip() for m in _FALLBACK_MARKERS if m in text)
        except Exception:
            pass  # watching must never break printing
        return _print(*args, **kwargs)

    _watching_print._workshop = True
    _builtins.print = _watching_print
_ip.events.register("pre_run_cell", workshop._pre)
_ip.events.register("post_run_cell", workshop._post)
print("Workshop harness ready:", "WORKED EXAMPLE" if workshop.worked else "checking your code")


## Setup

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU. The lab also runs on a CPU, but training is then much slower; on a CPU runtime set `QUICK = True` in the next cell, which trains each model for a quarter of the steps.

PyTorch, NumPy and matplotlib are preinstalled on Colab, so there is nothing to install and no API key to set.

**How the lab works.** There are five exercises and one optional stretch section. Each exercise has:

1. a **Predict** question to answer before you run anything;
2. a `# TODO` cell where you write one short function, or the last lines of one;
3. a folded **Solution** cell (open it only after trying);
4. a **Checkpoint** cell that tells you whether your function is right.

The folded solution cells never replace your code: each checkpoint checks your own functions and says so. If you are stuck, `workshop.use_reference(N)` lets you go on with exercise N's reference solution, and the checkpoint then says it checked the reference. To run the whole lab as a worked example, tick `WORKED_EXAMPLE` in the harness cell at the top; a worked run shows how the lab goes, not that you did it.

The training loops, the data pipeline, the n-gram baseline and all plots are provided. The two training runs are the slow cells.

**Run time.** Not yet timed on a Colab T4. On the 4-core CPU the lab was built on, shared with other jobs, Run all took 72 minutes at full settings and 26 minutes with `QUICK = True`. A T4 is expected to be much faster, but that is an expectation, not a measurement.

In [ ]:
# Setup: imports, device and seeds. Everything here is preinstalled on Colab.
import json
import math
import os
import time
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# QUICK = True trains each model for a quarter of the steps: for a CPU runtime or a dry run.
# Its numbers are worse than the lab's and are not the baselines Lab 5 compares with.
QUICK = os.environ.get("NLP_LLMS_QUICK") == "1"

NAVY, ACCENT, GRAY = "#16324f", "#b3541e", "#8a8a8a"  # plot colors
LAB_START = time.time()
timings = {}  # section -> seconds, printed at the end
gpu = f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else ""
print(f"PyTorch {torch.__version__} · device {DEVICE}{gpu} · QUICK = {QUICK}")
if DEVICE == "cpu" and not QUICK:
    print("No GPU: switch to a T4 runtime, or set QUICK = True above to shorten training.")

## Data

The language-model corpus of the workshop's running thread: **Tiny Shakespeare**, the same file and the same split as Lab 1.

| | |
|---|---|
| Train / val / test | 1,000,000 / 55,000 / 60,394 characters, split by position |
| Tokens | characters; the vocabulary $V$ is the 65 characters of train |
| Compared with | Lab 1 (character n-gram) now, Lab 5 (transformer) next |

The next cell is the workshop's standard loading cell, the same in every lab (see `data/README.md` in the repository). It checks the file against a hash, so everyone gets the same bytes and the same split.

The rule for the whole workshop: fit on **train**, choose settings on **val**, report on **test** once per model.

In [ ]:
import csv, gzip, hashlib, io, os, urllib.request
from pathlib import Path


def fetch(name, urls, sha256):
    """Return the bytes of a workshop data file, checked against its SHA-256."""
    cache = Path(os.environ.get("NLP_LLMS_DATA", "data"))
    for path in (cache / name, Path("../data") / name):
        if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == sha256:
            return path.read_bytes()
    for url in urls:  # canonical URL first, fallback second
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                blob = response.read()
        except OSError as error:
            print(f"Could not fetch {url}: {error}")
            continue
        if hashlib.sha256(blob).hexdigest() != sha256:
            print(f"Ignoring {url}: the file does not match the expected hash")
            continue
        cache.mkdir(parents=True, exist_ok=True)
        (cache / name).write_bytes(blob)
        return blob
    raise RuntimeError(f"Could not load {name} from any source")


TOPIC_CLASSES = ["cs.CL", "cs.CV", "cs.LG", "cs.RO"]  # label 0, 1, 2, 3
TOPIC_NAMES = ["Language", "Vision", "Machine learning", "Robotics"]


def load_topics():
    """arXiv Topics v1. Returns {"train" | "val" | "test": (texts, labels)}."""
    blob = fetch(
        "arxiv_topics_v1.csv.gz",
        [
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/arxiv_topics_v1.csv.gz",
            "https://cdn.jsdelivr.net/gh/project-delphi/nlp-llms@main/data/arxiv_topics_v1.csv.gz",
        ],
        "49b51502fc6cf6ecff23fa266c5372594a88414dd96513579237792710246dc5",
    )
    splits = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for row in csv.DictReader(io.StringIO(gzip.decompress(blob).decode("utf-8"))):
        texts, labels = splits[row["split"]]
        texts.append(row["title"] + "\n" + row["abstract"])
        labels.append(int(row["label"]))
    return splits


def load_lm_corpus():
    """Tiny Shakespeare. Returns {"train" | "val" | "test": str}, split by character offset."""
    blob = fetch(
        "tinyshakespeare.txt",
        [
            "https://raw.githubusercontent.com/karpathy/char-rnn/6f9487a6fe5b420b7ca9afb0d7c078e37c1d1b4e/data/tinyshakespeare/input.txt",
            "https://raw.githubusercontent.com/project-delphi/nlp-llms/main/data/tinyshakespeare.txt",
        ],
        "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",
    )
    text = blob.decode("utf-8")
    return {"train": text[:1_000_000], "val": text[1_000_000:1_055_000], "test": text[1_055_000:]}

In [ ]:
corpus = load_lm_corpus()  # {"train" | "val" | "test": str}
chars = sorted(set(corpus["train"]))  # the vocabulary V: the characters of train
char_to_id = {ch: i for i, ch in enumerate(chars)}
V = len(chars)  # |V|
assert V == 65
for split in ("val", "test"):
    assert set(corpus[split]) <= set(chars), f"{split} contains a character outside V"

ids = {split: torch.tensor([char_to_id[ch] for ch in text]) for split, text in corpus.items()}
# The splits are consecutive pieces of one text. The context of the first characters of a split
# is the text just before it, as in Lab 1.
history = {"val": ids["train"], "test": torch.cat([ids["train"], ids["val"]])}

for split, t in ids.items():
    print(f"{split:<5} {len(t):>9,} characters")
print(f"|V| = {V}: {''.join(chars)!r}")
print("\nStart of the test split:\n" + corpus["test"][:200])

# Part A · Recurrent Networks

## Exercise 1 · An RNN Cell by Hand (8 minutes)

A recurrent neural network reads one token at a time and updates a hidden state with the same function at every step (@eq-rnn in the briefing):

$$
a_t = W h_{t-1} + U x_t + b, \qquad h_t = \tanh(a_t).
$$

$x_t \in \mathbb{R}^d$ is the embedding of the token at step $t$, $h_t \in \mathbb{R}^{d_h}$ the hidden state, $W$ ($d_h \times d_h$) the recurrent matrix, $U$ ($d_h \times d$) the input matrix and $b$ the bias. The code works on a batch of $B$ sequences at once, one sequence per row, so $W h_{t-1}$ for every row is `h_prev @ W.T`.

| Argument | Shape |
|---|---|
| `x_t` | `(B, d)` |
| `h_prev` | `(B, d_h)` |
| `W`, `U`, `b` | `(d_h, d_h)`, `(d_h, d)`, `(d_h,)` |

**Predict.** For $B = 3$ and $d_h = 5$, what is the shape of $h_t$, and in what range do its entries lie? What is $h_t$ if $W$, $U$ and $b$ are all zero?

**Task.** Complete `rnn_cell_step`.

In [ ]:
# TODO 1: complete this function.
def rnn_cell_step(x_t, h_prev, W, U, b):
    """One RNN step, h_t = tanh(W h_prev + U x_t + b), for a batch. Returns (B, d_h)."""
    raise NotImplementedError("TODO 1")

In [ ]:
#@title Solution 1 — try it yourself first { display-mode: "form" }
@workshop.solution(1)
def rnn_cell_step(x_t, h_prev, W, U, b):
    """One RNN step, h_t = tanh(W h_prev + U x_t + b), for a batch. Returns (B, d_h)."""
    a_t = h_prev @ W.T + x_t @ U.T + b  # (B, d_h): W h_{t-1} + U x_t + b for every row
    return torch.tanh(a_t)  # h_t: (B, d_h), entries in (-1, 1)

<details>
<summary>Why this solution works</summary>

This is @eq-rnn. With one sequence per row, the column-vector product $W h_{t-1}$ becomes the row-vector product `h_prev @ W.T`, computed for all $B$ rows at once; `b` broadcasts over the rows. $\tanh$ squashes every entry into $(-1, 1)$, and with all-zero parameters $h_t = \tanh(0) = 0$.

PyTorch's `nn.RNNCell` computes the same thing but keeps two bias vectors, `bias_ih` and `bias_hh`. The checkpoint passes their sum as `b`. Note that `W`, `U` and `b` have no time index: the model below calls this function with the same parameters at every position.

</details>

In [ ]:
# Checkpoint 1: compare with PyTorch's nn.RNNCell given the same weights.
workshop.checkpoint(1)
torch.manual_seed(1)
_cell = nn.RNNCell(input_size=4, hidden_size=5)  # tanh nonlinearity by default
_b = _cell.bias_ih + _cell.bias_hh  # nn.RNNCell has two bias vectors; the briefing has one
_x1, _x2, _h0 = torch.randn(3, 4), torch.randn(3, 4), torch.randn(3, 5)
with torch.no_grad():
    _h1 = rnn_cell_step(_x1, _h0, _cell.weight_hh, _cell.weight_ih, _b)
    assert _h1 is not None, "rnn_cell_step returned None"
    assert _h1.shape == (3, 5), f"h_t must have shape (B, d_h) = (3, 5), got {tuple(_h1.shape)}"
    assert torch.allclose(_h1, _cell(_x1, _h0), atol=1e-5), (
        f"your step differs from nn.RNNCell by up to {(_h1 - _cell(_x1, _h0)).abs().max().item():.2e}: "
        "with one sequence per row, W h_prev is h_prev @ W.T, and tanh comes last")
    _h2 = rnn_cell_step(_x2, _h1, _cell.weight_hh, _cell.weight_ih, _b)  # the state feeds step 2
    assert torch.allclose(_h2, _cell(_x2, _cell(_x1, _h0)), atol=1e-5), "two steps differ"
print("Checkpoint 1 passed")

**The RNN language model** (provided). The model below is @eq-rnn and @eq-output of the briefing: an embedding gives $x_t$, your `rnn_cell_step` runs once per position, and a linear layer turns each $h_t$ into logits $z_t$ over the next character. It is a Python loop over time steps, the clearest form and fast enough for 128-character chunks. The LSTM in Part B uses PyTorch's fused `nn.LSTM` instead.

In [ ]:
class RNNLM(nn.Module):
    """Character RNN language model built on your rnn_cell_step."""

    def __init__(self, vocab_size, d=64, d_h=256):
        super().__init__()
        self.d_h = d_h
        self.embed = nn.Embedding(vocab_size, d)  # x_t = e_{w_t}
        bound = 1 / math.sqrt(d_h)  # the same initialization as PyTorch's nn.RNN
        self.W = nn.Parameter(torch.empty(d_h, d_h).uniform_(-bound, bound))
        self.U = nn.Parameter(torch.empty(d_h, d).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.zeros(d_h))
        self.out = nn.Linear(d_h, vocab_size)  # z_t = W_o h_t + b_o

    def forward(self, x, h=None):
        """x: (B, T) character IDs. Returns logits (B, T, |V|) and the last state h_T (B, d_h)."""
        e = self.embed(x)  # (B, T, d)
        if h is None:
            h = e.new_zeros(x.shape[0], self.d_h)  # h_0 = 0
        hs = []
        for t in range(x.shape[1]):
            h = rnn_cell_step(e[:, t], h, self.W, self.U, self.b)  # (B, d_h)
            hs.append(h)
        return self.out(torch.stack(hs, dim=1)), h


def n_params(model):
    return sum(p.numel() for p in model.parameters())

## Exercise 2 · Loss, Perplexity and Bits per Character (6 minutes)

The model reads $w_1, \dots, w_T$ and is trained to predict $w_2, \dots, w_{T+1}$, the same text shifted by one position. The loss is the mean cross-entropy, and perplexity is its exponential (@eq-loss and @eq-ppl):

$$
\mathcal{L} = \frac{1}{T}\sum_{t=1}^{T} -\log \hat{y}_t[w_{t+1}], \qquad \hat{y}_t = \mathrm{softmax}(z_t), \qquad \mathrm{PPL} = \exp(\mathcal{L}), \qquad \mathrm{BPC} = \frac{\mathcal{L}}{\ln 2}.
$$

The logarithm is natural, so $\mathcal{L}$ is in **nats** per character. **Bits per character** (BPC) is the same quantity in base 2.

This one function scores every model in this lab, the n-gram baseline included, and it is the training loss. Lab 5 reuses it unchanged.

**Predict.** A model outputs the same logit for all 65 characters. What are its loss, its perplexity and its bits per character?

**Task.** Complete `lm_loss_and_ppl(logits, targets)`. `logits` has shape `(..., |V|)` and `targets` the matching shape `(...)`. Return three tensors: the mean loss (it must stay differentiable, because the training loop calls `backward()` on it), the perplexity and the bits per character. `F.log_softmax` and `Tensor.gather` are enough; `F.cross_entropy` also works.

<details>
<summary>Hint for TODO 2</summary>

Take the log-softmax over the last axis, the vocabulary, pick each target's entry, and average over every predicted position, all $B \cdot T$ of them, not over the batch alone (briefing @eq-loss). Stay in nats, with the natural log; BPC divides the nats by $\ln 2$ afterwards. Equal logits must give a loss of $\ln 65 \approx 4.17$ and a perplexity of exactly 65.

</details>

In [ ]:
# TODO 2: complete this function.
def lm_loss_and_ppl(logits, targets):
    """Mean NLL in nats, perplexity and bits per character. logits (..., |V|), targets (...)."""
    raise NotImplementedError("TODO 2")

In [ ]:
#@title Solution 2 — try it yourself first { display-mode: "form" }
@workshop.solution(2)
def lm_loss_and_ppl(logits, targets):
    """Mean NLL in nats, perplexity and bits per character. logits (..., |V|), targets (...)."""
    logits = logits.reshape(-1, logits.shape[-1])  # (N, |V|): one row per predicted position
    targets = targets.reshape(-1)  # (N,)
    log_probs = F.log_softmax(logits, dim=-1)  # log ŷ_t, (N, |V|)
    loss = -log_probs.gather(1, targets[:, None]).mean()  # mean of -log ŷ_t[w_{t+1}], nats
    return loss, loss.exp(), loss / math.log(2)

<details>
<summary>Why this solution works</summary>

`log_softmax` gives $\log \hat{y}_t$ for every position and character; `gather` picks the entry of the character that actually came next; the mean over positions is @eq-loss. Perplexity is @eq-ppl, and dividing nats by $\ln 2$ converts them to bits.

With equal logits, $\hat{y}_t$ is uniform, every target has probability $1/|V|$, the loss is $\ln |V| = \ln 65 \approx 4.17$ nats, the perplexity is exactly $|V| = 65$ and the BPC is $\log_2 65 \approx 6.02$. That is the "effective branching factor" reading of perplexity from Lab 1.

Flattening the leading dimensions means the same call works for a training batch `(B, T, |V|)` and for a whole test split `(N, |V|)`.

</details>

In [ ]:
# Checkpoint 2
workshop.checkpoint(2)
def _num(v):
    return v.detach().item() if torch.is_tensor(v) else float(v)


_out = lm_loss_and_ppl(torch.zeros(10, V), torch.randint(0, V, (10,)))  # equal logits
assert _out is not None and len(_out) == 3, "return (loss, ppl, bpc)"
_loss, _ppl, _bpc = _out
assert math.isclose(_num(_loss), math.log(V), rel_tol=1e-6), (
    f"equal logits: loss must be ln|V| = {math.log(V):.4f} nats, got {_num(_loss)}: "
    "use the natural log and average over the positions")
assert math.isclose(_num(_ppl), V, rel_tol=1e-5), "equal logits: perplexity must be |V| = 65"
assert math.isclose(_num(_bpc), math.log2(V), rel_tol=1e-6), "BPC = loss / ln 2"

torch.manual_seed(2)
_z = torch.randn(4, 7, V, requires_grad=True)  # a batch shaped like the training batches
_y = torch.randint(0, V, (4, 7))
_loss, _ppl, _bpc = lm_loss_and_ppl(_z, _y)
assert torch.allclose(_loss, F.cross_entropy(_z.reshape(-1, V), _y.reshape(-1))), (
    f"the loss {_num(_loss):.4f} differs from F.cross_entropy, "
    f"{F.cross_entropy(_z.reshape(-1, V), _y.reshape(-1)).item():.4f}: average over all B x T positions")
assert _loss.requires_grad, "keep the loss differentiable: the training loop calls backward() on it"
assert math.isclose(_num(_ppl), math.exp(_num(_loss)), rel_tol=1e-5), (
    f"perplexity must be exp(loss) = {math.exp(_num(_loss)):.4f}, got {_num(_ppl)}")
print("Checkpoint 2 passed")

## Run · The Baseline, Measured the Same Way (2 minutes)

Perplexity is a per-token number, so it can only be compared at the same token unit. Lab 1's word-level perplexities say nothing about a character model. The comparison here is character-level on both sides, and the notebook enforces each rule of the briefing:

| Rule | How it is enforced |
|---|---|
| Same token unit | The baseline is a *character* n-gram, built with Lab 1's method (counts and add-$k$) |
| Same data | `load_lm_corpus()`: the same file, checked by hash, and the same split |
| Same vocabulary | All models predict over the 65 characters of train (asserted in the data cell) |
| Same scored positions | Every character of the split is scored; its left context comes from the text before the split |
| Same formula | Every model is scored by your `lm_loss_and_ppl` |
| No tuning on test | $k$ and all neural settings are chosen on val; test is scored once per model, at the end |

Notebooks cannot import from one another on Colab, so the next cell restates Lab 1's add-$k$ n-gram for characters. It is the same model as Lab 1's `ngram_counts` and `prob`, vectorized with NumPy so that it scores a whole split against all 65 possible next characters at once. Its log-probabilities go straight into `lm_loss_and_ppl`: a vector of log-probabilities is a valid vector of logits, because $\mathrm{softmax}(\log p) = p$.

In [ ]:
# Lab 1's method, applied to characters: counts from train only, add-k smoothing.
def window_codes(seq, length, base):
    """One integer per window of `length` consecutive IDs: the window read as base-`base` digits."""
    out = np.zeros(len(seq) - length + 1, dtype=np.int64)
    for j in range(length):
        out = out * base + seq[j : len(seq) - length + 1 + j]
    return out


def lookup(keys, values, queries):
    """The value stored for each query in the sorted array `keys`, or 0 if absent (like a Counter)."""
    idx = np.clip(np.searchsorted(keys, queries), 0, len(keys) - 1)
    return np.where(keys[idx] == queries, values[idx], 0)


def char_ngram_counts(n, split):
    """c(context, w) for all 65 w, shape (N, |V|), and c(context), shape (N, 1), per position."""
    base, bos = V + 1, V  # IDs 0..64, plus a start ID that only pads the start of train
    train = np.concatenate([np.full(n - 1, bos), ids["train"].numpy()])
    grams = window_codes(train, n, base)  # every training n-gram: the context digits, then w
    gram_keys, gram_counts = np.unique(grams, return_counts=True)
    ctx_keys, ctx_counts = np.unique(grams // base, return_counts=True)  # drop w: the context
    before = history[split].numpy()[len(history[split]) - (n - 1) :]  # n - 1 characters of context
    stream = np.concatenate([before, ids[split].numpy()])
    context = window_codes(stream, n - 1, base)[: len(ids[split])]  # (N,) context of each position
    counts = lookup(gram_keys, gram_counts, context[:, None] * base + np.arange(V))  # (N, |V|)
    return counts, lookup(ctx_keys, ctx_counts, context)[:, None]


def add_k_log_probs(counts, totals, k):
    """log P_add-k(w | context) = log((c(context, w) + k) / (c(context) + k|V|)), (N, |V|)."""
    return torch.from_numpy(np.log((counts + k) / (totals + k * V)))


def score(logits, targets):
    """Score with your lm_loss_and_ppl; returns plain floats."""
    loss, ppl, bpc = lm_loss_and_ppl(logits, targets)
    return {"nats": float(loss), "ppl": float(ppl), "bpc": float(bpc)}

In [ ]:
K_GRID = [0.001, 0.01, 0.1, 0.5, 1.0]  # Lab 1's grid
NGRAM_ORDERS = [2, 3, 5]  # bigram, trigram, and the best order n <= 5 on val in Lab 1
started = time.time()
results, ngram_k = {}, {}  # model name -> {"val": metrics, "test": metrics}; n -> chosen k
for n in NGRAM_ORDERS:
    counts_val = char_ngram_counts(n, "val")
    val = {k: score(add_k_log_probs(*counts_val, k), ids["val"]) for k in K_GRID}
    ngram_k[n] = min(K_GRID, key=lambda k: val[k]["nats"])  # k is chosen on val
    test = score(add_k_log_probs(*char_ngram_counts(n, "test"), ngram_k[n]), ids["test"])
    results[f"{n}-gram"] = {"val": val[ngram_k[n]], "test": test}
timings["n-gram baseline"] = time.time() - started

# The same numbers as Lab 1 (data/baselines.json), so the protocol is the same.
LAB1_TEST_NATS = {2: 2.4922, 3: 2.0859, 5: 1.8326}
LAB1_K = {2: 0.1, 3: 0.1, 5: 0.01}
for n in NGRAM_ORDERS:
    assert ngram_k[n] == LAB1_K[n], (n, ngram_k[n])
    assert abs(results[f"{n}-gram"]["test"]["nats"] - LAB1_TEST_NATS[n]) < 1e-4, n
print("Reproduces Lab 1's character n-gram baselines.\n")
print("Character n-gram, add-k, validation split (test is scored once, in the results table)")
print(" model      k   val nats/char   val PPL   val BPC")
for n in NGRAM_ORDERS:
    m = results[f"{n}-gram"]["val"]
    print(f"{n}-gram  {ngram_k[n]:>5}  {m['nats']:>14.4f}  {m['ppl']:>8.3f}  {m['bpc']:>8.4f}")
print(f"({timings['n-gram baseline']:.1f} s)")

## Run · Train the RNN Language Model (4 minutes)

The training loop and the evaluation routine are provided.

- **Training** (`train_lm`). Each step takes 64 random chunks of $T = 128$ characters from train, runs the model from $h_0 = 0$ with **teacher forcing** (the inputs are the true characters), computes your `lm_loss_and_ppl`, backpropagates within the chunk only (**truncated backpropagation through time**), clips the gradient norm at $\kappa = 1$ and takes an Adam step.
- **Evaluation** (`evaluate_lm`). The model reads the split as one stream, in chunks of 1,000 characters, carrying the hidden state from chunk to chunk. Before the first scored character it reads the 1,000 characters that precede the split, without scoring them, so no position is predicted from a cold state. Every character of the split is scored exactly once.

**Predict.** On validation, which will be lowest: the bigram, the trigram, Lab 1's best 5-gram or the RNN? The RNN sees all of the preceding text, but trains for only a few minutes.

In [ ]:
def get_batch(split_ids, batch_size, T, generator):
    """batch_size random chunks of T + 1 characters: inputs x (B, T) and targets y (B, T)."""
    starts = torch.randint(0, len(split_ids) - T - 1, (batch_size,), generator=generator)
    chunk = torch.stack([split_ids[s : s + T + 1] for s in starts.tolist()])  # (B, T + 1)
    return chunk[:, :-1].to(DEVICE), chunk[:, 1:].to(DEVICE)  # targets: inputs shifted by one


def train_lm(model, steps, lr, kappa=1.0, batch_size=64, T=128, seed=SEED):
    """Adam with truncated BPTT and gradient clipping. Returns per-step loss and gradient norm."""
    generator = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses, norms = [], []
    model.train()
    started = time.time()
    for step in range(1, steps + 1):
        x, y = get_batch(ids["train"], batch_size, T, generator)
        logits, _ = model(x)  # (B, T, |V|), from h_0 = 0
        loss, _, _ = lm_loss_and_ppl(logits, y)
        optimizer.zero_grad()
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), kappa)  # @eq-clip
        optimizer.step()
        losses.append(loss.detach())
        norms.append(norm.detach())
        if step % max(1, steps // 5) == 0:
            print(f"  step {step:>5}/{steps}  train loss {loss.item():.3f}  "
                  f"grad norm {norm.item():.2f}  {time.time() - started:>5.0f} s")
    return {"loss": torch.stack(losses).tolist(), "grad_norm": torch.stack(norms).tolist()}


@torch.no_grad()
def evaluate_lm(model, split, chunk=1000):
    """Score every character of `split`, carrying the state across chunks. Returns metrics."""
    model.eval()
    warmup = history[split][-chunk:]  # read first, never scored
    stream = torch.cat([warmup, ids[split]]).to(DEVICE)
    x, y = stream[:-1], stream[1:]  # predict stream[i + 1] from stream[: i + 1]
    state, logits = None, []
    for s in range(0, len(x), chunk):
        z, state = model(x[None, s : s + chunk], state)  # z: (1, L, |V|)
        logits.append(z[0])
    logits, targets = torch.cat(logits)[len(warmup) - 1 :], y[len(warmup) - 1 :]
    assert len(targets) == len(ids[split]), "every character of the split, exactly once"
    model.train()
    return score(logits, targets)

In [ ]:
# Settings sized for a free T4. Sizes, learning rate and kappa are the lab brief's starting point;
# nothing was tuned on test.
D, D_H, LR, KAPPA = 64, 256, 2e-3, 1.0
STEPS = {"RNN": 1000, "LSTM": 2000}
if QUICK:
    STEPS = {name: steps // 4 for name, steps in STEPS.items()}

torch.manual_seed(SEED)
rnn = RNNLM(V, D, D_H).to(DEVICE)
print(f"RNN LM, {n_params(rnn):,} parameters, {STEPS['RNN']} steps")
started = time.time()
rnn_log = train_lm(rnn, STEPS["RNN"], LR, KAPPA)
timings["RNN training"] = time.time() - started
results["RNN"] = {"val": evaluate_lm(rnn, "val")}
timings["RNN evaluation (val)"] = time.time() - started - timings["RNN training"]

print("\n model     val nats/char   val PPL   val BPC")
for name in ["2-gram", "3-gram", "5-gram", "RNN"]:
    m = results[name]["val"]
    print(f" {name:<7} {m['nats']:>14.4f}  {m['ppl']:>8.3f}  {m['bpc']:>8.4f}")
print(f"(training {timings['RNN training']:.0f} s, evaluation {timings['RNN evaluation (val)']:.0f} s)")

**Explain.** Compare with your prediction. The RNN conditions on the whole prefix through $h_t$, yet it was trained on 128-character chunks for a few minutes. Look at where it lands relative to the bigram, the trigram and the 5-gram, which conditions on exactly four characters. Exercise 3 measures how far back the RNN's gradient actually reaches.

## Exercise 3 · Vanishing and Exploding Gradients (9 minutes)

### Part A · How Far Back Does the Gradient Reach?

The gradient that teaches the RNN a long-range dependency passes through the product of Jacobians (@eq-jacprod and @eq-bound):

$$
\frac{\partial h_t}{\partial h_s} = \prod_{j=s+1}^{t} \mathrm{diag}(1 - h_j^2)\, W, \qquad \Big\lVert \frac{\partial h_t}{\partial h_s} \Big\rVert \le \lVert W \rVert^{\,t-s}.
$$

The next cell measures it on the RNN you just trained. It runs the model over 64 validation chunks of 200 characters (longer than the training chunks), takes the loss $\mathcal{L}_T$ at the **last** position only, backpropagates, and records $\lVert \partial \mathcal{L}_T / \partial h_s \rVert$ for every earlier position $s$, averaged over the batch. One backward pass, no training.

**Predict.** Sketch the curve of $\lVert \partial \mathcal{L}_T / \partial h_s \rVert$ against the distance $T - s$ on a log scale. Is it flat, a straight line going down, or a straight line going up? Roughly how many characters back will it have fallen by a factor of 1,000?

In [ ]:
def grad_norm_by_distance(step, state, embed, out, x, y):
    """Mean over the batch of ||dL_T/dh_s||, indexed by the distance T - s (index 0 is s = T).

    step(x_t, state) -> (h_t, new_state) runs one recurrent step; L_T is the loss at the last position.
    """
    e = embed(x)  # (B, L, d)
    hs = []
    for t in range(x.shape[1]):
        h, state = step(e[:, t], state)
        h.retain_grad()  # keep dL_T/dh_t, which autograd would otherwise discard
        hs.append(h)
    loss_T = F.cross_entropy(out(hs[-1]), y[:, -1])  # only the last position's loss
    loss_T.backward()
    return torch.stack([h.grad.norm(dim=1).mean() for h in reversed(hs)]).cpu()


def rnn_step(x_t, h):
    h = rnn_cell_step(x_t, h, rnn.W, rnn.U, rnn.b)
    return h, h


started = time.time()
DIST_LEN = 200
dist_x, dist_y = get_batch(ids["val"], 64, DIST_LEN, torch.Generator().manual_seed(SEED))
rnn_dist = grad_norm_by_distance(rnn_step, torch.zeros(64, D_H, device=DEVICE),
                                 rnn.embed, rnn.out, dist_x, dist_y)
rnn.zero_grad()
spectral_W = torch.linalg.matrix_norm(rnn.W.detach(), ord=2).item()  # ||W||, the spectral norm


def plot_grad_distance(curves):
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for label, norms, color in curves:
        norms = norms.numpy()
        ax.semilogy(np.arange(len(norms)), np.where(norms > 0, norms, np.nan), color=color,
                    linewidth=2, label=label)
    ax.set_xlabel("distance T − s (characters)")
    ax.set_ylabel("‖∂L_T / ∂h_s‖ (mean over 64 chunks)")
    ax.legend(frameon=False)
    plt.show()


plot_grad_distance([("RNN", rnn_dist, NAVY)])
print(f"Spectral norm of the trained W: ||W|| = {spectral_W:.2f}")
for d in (0, 1, 10, 25, 50, 100, DIST_LEN - 1):
    print(f"  distance {d:>3}: {rnn_dist[d]:.2e}")
if (rnn_dist == 0).any():
    print(f"  From distance {int((rnn_dist == 0).nonzero()[0])} on, the gradient underflows to exactly 0 "
          "in float32 (not drawn on the log axis).")
timings["Gradient measurement (RNN)"] = time.time() - started

**Explain.** The curve is close to a straight line going down on a log scale: the gradient shrinks by a roughly constant factor per character, which is exponential decay, as the product of Jacobians predicts. Compare $\lVert W \rVert$ with 1. If it is above 1, the bound $\lVert W \rVert^{t-s}$ does not force decay, so the decay comes from the actual one-step Jacobians, and there the factor $\mathrm{diag}(1 - h_j^2)$ matters: every saturated $\tanh$ unit multiplies the gradient passing back through it by nearly 0. Either way, a character 50 positions back gets almost no say in the update, so the RNN learns mostly short-range patterns. Nothing crashes; the model is just quietly short-sighted.

In [ ]:
# Checkpoint 3A: the measured gradient decays with distance.
# In every build run the gradient underflowed to exactly 0 (from distance 138 at full settings, 91-107 with QUICK): an infinite factor.
workshop.checkpoint(label="3A")
RNN_DECAY_MIN = 1e4
decay = (rnn_dist[1] / rnn_dist[-1]).item() if rnn_dist[-1] > 0 else math.inf
print(f"||dL_T/dh_s|| at distance 1 is {decay:.1e} times its value at distance {DIST_LEN - 1}")
assert decay > RNN_DECAY_MIN, "the RNN's gradient should vanish over 200 characters"
print("Checkpoint 3A passed")

### Part B · Gradient Clipping

The opposite failure: when the product of Jacobians grows, one gradient can be enormous and a single update throws the parameters far away. **Clipping by norm** (@eq-clip) rescales the whole gradient $g$, all parameters flattened into one vector, when it is longer than a threshold $\kappa$:

$$
g \leftarrow g \cdot \min\Big(1, \frac{\kappa}{\lVert g \rVert}\Big).
$$

**Predict.** After clipping a gradient of norm 10 at $\kappa = 1$, what is its norm, and what is the cosine between the old and the new gradient? What happens to a gradient of norm 0.5?

**Task.** Complete `clip_gradients(params, kappa)`. It changes each `p.grad` in place and returns the global norm before clipping, like `torch.nn.utils.clip_grad_norm_`. (The training loop above already uses PyTorch's version; the demonstration below uses yours.)

<details>
<summary>Hint for TODO 3</summary>

Compute one norm over all the gradients together: the square root of the sum of squares across every tensor (briefing @eq-clip). Then scale every gradient by the same factor $\min(1, \kappa / \lVert g \rVert)$: clipping each tensor by its own norm changes the direction, and leaving out the $\min$ makes a small gradient longer. Return the norm you measured before scaling.

</details>

In [ ]:
# TODO 3: complete this function.
def clip_gradients(params, kappa):
    """Scale every p.grad in place so that the global norm is at most kappa; return the old norm."""
    grads = [p.grad for p in params if p.grad is not None]
    ...

In [ ]:
#@title Solution 3 — try it yourself first { display-mode: "form" }
@workshop.solution(3)
def clip_gradients(params, kappa):
    """Scale every p.grad in place so that the global norm is at most kappa; return the old norm."""
    grads = [p.grad for p in params if p.grad is not None]
    norm = torch.sqrt(sum((g**2).sum() for g in grads))  # ||g|| over all parameters together
    scale = torch.clamp(kappa / (norm + 1e-6), max=1.0)  # min(1, kappa / ||g||)
    for g in grads:
        g.mul_(scale)  # in place: the optimizer reads p.grad
    return norm

<details>
<summary>Why this solution works</summary>

The norm is taken over **all** parameters together, so clipping rescales the whole gradient by one factor and keeps its direction: the cosine between the old and the new gradient is 1. A gradient of norm 10 at $\kappa = 1$ comes out with norm 1; one of norm 0.5 is multiplied by $\min(1, 2) = 1$ and is unchanged. The small `1e-6` avoids a division by zero and matches PyTorch.

Clipping each parameter separately would change the direction. Clipping caps the size of an update; it does nothing for the vanishing gradient of Part A, where the long-range terms are tiny *relative* to the short-range terms inside the same vector.

</details>

In [ ]:
# Checkpoint 3B
workshop.checkpoint(3, label="3B")
def _grads(scale, seed=3):
    torch.manual_seed(seed)
    params = [nn.Parameter(torch.randn(4, 5)), nn.Parameter(torch.randn(5))]
    for p in params:
        p.grad = torch.randn_like(p) * scale
    return params


def _flat(params):
    return torch.cat([p.grad.flatten() for p in params])


_ps = _grads(10.0)
_before = _flat(_ps).clone()
_norm = clip_gradients(_ps, kappa=1.0)
assert _norm is not None, "return the norm before clipping"
assert math.isclose(float(_norm), _before.norm().item(), rel_tol=1e-5), (
    f"return the norm BEFORE clipping: expected {_before.norm().item():.4f}, got {float(_norm):.4f}")
assert _flat(_ps).norm() <= 1.0 + 1e-5, (
    f"norm after clipping is {_flat(_ps).norm():.4f} > kappa = 1: take one norm over all the gradients, not one per tensor")
assert F.cosine_similarity(_before, _flat(_ps), dim=0) > 1 - 1e-6, "clipping must keep the direction"
_ps = _grads(0.01)
_before = _flat(_ps).clone()
clip_gradients(_ps, kappa=1.0)
assert torch.equal(_flat(_ps), _before), (
    f"a gradient already below kappa must be left unchanged; it was scaled by "
    f"{(_flat(_ps).norm() / _before.norm()).item():.4g}: the factor is min(1, kappa / norm)")
_ours, _ref = _grads(10.0), _grads(10.0)
_n_ours, _n_ref = clip_gradients(_ours, 0.5), torch.nn.utils.clip_grad_norm_(_ref, 0.5)
assert math.isclose(float(_n_ours), float(_n_ref), rel_tol=1e-5), (
    f"returned norm {float(_n_ours):.4f}, but clip_grad_norm_ returned {float(_n_ref):.4f}")
assert torch.allclose(_flat(_ours), _flat(_ref)), "differs from torch.nn.utils.clip_grad_norm_"
print("Checkpoint 3B passed")

**Run: explosion with and without clipping.** Two fresh RNNs train for 150 steps with plain SGD at a deliberately high learning rate (2.0), on identical batches. The only difference is your `clip_gradients` at $\kappa = 1$. (Adam, used everywhere else, rescales its steps and hides most of this effect, so the demonstration uses SGD.)

**Predict.** What will the gradient norm and the loss do in the run without clipping?

In [ ]:
def sgd_run(kappa, steps=150, lr=2.0, batch_size=32, T=64, seed=SEED):
    """Fresh RNN, plain SGD, your clip_gradients. Returns per-step gradient norm and loss."""
    torch.manual_seed(seed)
    model = RNNLM(V, D, D_H).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    generator = torch.Generator().manual_seed(seed)  # the same batches in both runs
    norms, losses = [], []
    for _ in range(steps):
        x, y = get_batch(ids["train"], batch_size, T, generator)
        loss, _, _ = lm_loss_and_ppl(model(x)[0], y)
        optimizer.zero_grad()
        loss.backward()
        norms.append(float(clip_gradients(model.parameters(), kappa)))  # norm before clipping
        losses.append(loss.item())
        optimizer.step()
    return {"grad_norm": norms, "loss": losses}


started = time.time()
clip_demo = {"no clipping": sgd_run(kappa=math.inf), "clipping at κ = 1": sgd_run(kappa=1.0)}
timings["Clipping demonstration"] = time.time() - started

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for (label, run), color in zip(clip_demo.items(), [ACCENT, NAVY]):
    axes[0].semilogy(run["grad_norm"], color=color, label=label)
    axes[1].plot(run["loss"], color=color, label=label)
axes[0].axhline(1.0, color=GRAY, linestyle="--", linewidth=1)
axes[0].set(xlabel="step", ylabel="gradient norm before clipping")
axes[1].set(xlabel="step", ylabel="training loss (nats/char)")
axes[1].legend(frameon=False)
plt.show()
for label, run in clip_demo.items():
    print(f"{label:<18} largest gradient norm {max(run['grad_norm']):>9.1f}   largest loss "
          f"{max(run['loss']):>6.2f}   mean loss over the last 20 steps {np.mean(run['loss'][-20:]):.3f}")
_unclipped = np.mean(clip_demo["no clipping"]["loss"][-20:])
_clipped = np.mean(clip_demo["clipping at κ = 1"]["loss"][-20:])
assert _unclipped > math.log(V), "without clipping this run should end worse than a uniform guess"
assert _clipped < _unclipped - 1.0, "with clipping the same run should train"

**Explain.** Without clipping, the gradient norm reaches values in the hundreds or more, the loss jumps far above the $\ln 65 \approx 4.2$ of a uniform guess, and the run does not recover within 150 steps: single huge updates destroyed what had been learned. With clipping, every update has length at most $\text{lr} \cdot \kappa$, and the same learning rate trains normally. This is why the training loop clips at $\kappa = 1$ for both models.

# Part B · The LSTM

## Exercise 4 · The LSTM Cell (6 minutes)

The LSTM adds a **cell state** $c_t$ that is updated *additively* and three **gates** with entries in $(0, 1)$ (@eq-lstm):

$$
\begin{aligned}
f_t &= \sigma(W_f h_{t-1} + U_f x_t + b_f), \quad i_t = \sigma(W_i h_{t-1} + U_i x_t + b_i), \quad o_t = \sigma(W_o h_{t-1} + U_o x_t + b_o), \\
\tilde{c}_t &= \tanh(W_c h_{t-1} + U_c x_t + b_c), \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t, \qquad h_t = o_t \odot \tanh(c_t).
\end{aligned}
$$

($W_o$ and $b_o$ here are the output *gate's* parameters, written $W_o^{\text{g}}$ and $b_o^{\text{g}}$ in the briefing.) The four gate lines are given; you write the last two.

PyTorch packs the four weight matrices into one, in the order input, forget, cell (it calls the candidate $g$), output. The helper `lstm_params` slices them apart into the briefing's names.

**Predict.** For one unit, suppose $f_t = 1$ and $i_t = 0$. What is $c_t$? And if $f_t = 0$ and $i_t = 1$?

**Task.** Write the cell update and the hidden state.

In [ ]:
def lstm_params(weight_ih, weight_hh, bias_ih, bias_hh):
    """Split PyTorch's packed LSTM weights (gate order i, f, g, o) into W_*, U_*, b_*."""
    d_h = weight_hh.shape[1]
    p = {}
    for k, gate in enumerate(["i", "f", "c", "o"]):  # PyTorch's "g" is the candidate c~
        rows = slice(k * d_h, (k + 1) * d_h)
        p[f"W_{gate}"] = weight_hh[rows]  # (d_h, d_h)
        p[f"U_{gate}"] = weight_ih[rows]  # (d_h, d)
        p[f"b_{gate}"] = bias_ih[rows] + bias_hh[rows]  # two biases in PyTorch, one in the briefing
    return SimpleNamespace(**p)

In [ ]:
# TODO 4: write the last two lines (the four gate lines are given).
def lstm_cell_step(x_t, h_prev, c_prev, p):
    """One LSTM step. p holds W_*, U_*, b_* for f, i, o, c. Returns (h_t, c_t), each (B, d_h)."""
    f = torch.sigmoid(h_prev @ p.W_f.T + x_t @ p.U_f.T + p.b_f)  # forget gate, (B, d_h)
    i = torch.sigmoid(h_prev @ p.W_i.T + x_t @ p.U_i.T + p.b_i)  # input gate
    o = torch.sigmoid(h_prev @ p.W_o.T + x_t @ p.U_o.T + p.b_o)  # output gate
    c_tilde = torch.tanh(h_prev @ p.W_c.T + x_t @ p.U_c.T + p.b_c)  # candidate values
    c = ...  # TODO: the cell update
    h = ...  # TODO: the hidden state
    return h, c

In [ ]:
#@title Solution 4 — try it yourself first { display-mode: "form" }
@workshop.solution(4)
def lstm_cell_step(x_t, h_prev, c_prev, p):
    """One LSTM step. p holds W_*, U_*, b_* for f, i, o, c. Returns (h_t, c_t), each (B, d_h)."""
    f = torch.sigmoid(h_prev @ p.W_f.T + x_t @ p.U_f.T + p.b_f)  # forget gate, (B, d_h)
    i = torch.sigmoid(h_prev @ p.W_i.T + x_t @ p.U_i.T + p.b_i)  # input gate
    o = torch.sigmoid(h_prev @ p.W_o.T + x_t @ p.U_o.T + p.b_o)  # output gate
    c_tilde = torch.tanh(h_prev @ p.W_c.T + x_t @ p.U_c.T + p.b_c)  # candidate values
    c = f * c_prev + i * c_tilde  # keep part of the old memory, write part of the candidate
    h = o * torch.tanh(c)  # reveal part of the squashed memory
    return h, c

<details>
<summary>Why this solution works</summary>

Both lines are elementwise (`*` is $\odot$). The cell update is a gated *sum*: with $f_t = 1$ and $i_t = 0$ the unit copies $c_{t-1}$ unchanged; with $f_t = 0$ and $i_t = 1$ it forgets everything and stores the candidate. That sum is why the gradient can survive: along the path $c_{s} \to c_{s+1} \to \dots \to c_t$ the Jacobian is just $\prod_j \mathrm{diag}(f_j)$ (@eq-lstm-jac), with no repeated $W$ and no $\tanh'$.

</details>

In [ ]:
# Checkpoint 4: compare with PyTorch's nn.LSTMCell given the same weights.
workshop.checkpoint(4)
torch.manual_seed(4)
_cell = nn.LSTMCell(input_size=4, hidden_size=5)
_p = lstm_params(_cell.weight_ih, _cell.weight_hh, _cell.bias_ih, _cell.bias_hh)
_x, _h, _c = torch.randn(3, 4), torch.randn(3, 5), torch.randn(3, 5)
with torch.no_grad():
    _out = lstm_cell_step(_x, _h, _c, _p)
    _h_ref, _c_ref = _cell(_x, (_h, _c))
assert _out[0] is not Ellipsis and _out[1] is not Ellipsis, "replace both ... with your code"
assert torch.allclose(_out[1], _c_ref, atol=1e-5), (
    f"c_t differs from nn.LSTMCell by up to {(_out[1] - _c_ref).abs().max().item():.2e}: "
    "the cell update is a gated sum of c_prev and the candidate, both gates applied elementwise")
assert torch.allclose(_out[0], _h_ref, atol=1e-5), (
    f"h_t differs from nn.LSTMCell by up to {(_out[0] - _h_ref).abs().max().item():.2e}: "
    "squash the new c_t with tanh, then gate it with o")
print("Checkpoint 4 passed")

## Run · Train the LSTM Language Model (7 minutes)

The model is the RNN language model with `nn.LSTM` in place of the loop over `rnn_cell_step`: the same embedding size, hidden size, output layer, training loop, data and number of characters per step. `nn.LSTM` runs @eq-lstm in one fused kernel; the cell first checks that it computes exactly what your `lstm_cell_step` computes. As in the briefing, the forget-gate bias starts at 1, so $f_t \approx \sigma(1) \approx 0.73$ at initialization and the gradient flows through the cell state from the first step.

It trains for more steps than the RNN because a fused `nn.LSTM` step is expected to be cheaper on a GPU than the RNN's Python loop. The comparison that matters is with the n-gram baseline; the RNN–LSTM gap below is measured at these settings, not at equal compute.

**Predict.** Will the LSTM beat Lab 1's best n-gram, the 5-gram, on test? By how much?

In [ ]:
class LSTMLM(nn.Module):
    """Character LSTM language model. nn.LSTM computes the same step as your lstm_cell_step."""

    def __init__(self, vocab_size, d=64, d_h=256):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d)
        self.lstm = nn.LSTM(d, d_h, batch_first=True)
        with torch.no_grad():
            self.lstm.bias_ih_l0[d_h : 2 * d_h].fill_(1.0)  # b_f = 1 (rows d_h..2d_h: forget gate)
            self.lstm.bias_hh_l0[d_h : 2 * d_h].zero_()
        self.out = nn.Linear(d_h, vocab_size)

    def forward(self, x, state=None):
        """x: (B, T). Returns logits (B, T, |V|) and the last state (h_T, c_T)."""
        hs, state = self.lstm(self.embed(x), state)  # hs: (B, T, d_h), every h_t
        return self.out(hs), state

    def params(self):
        """The briefing's W_*, U_*, b_* views of the trained weights."""
        m = self.lstm
        return lstm_params(m.weight_ih_l0, m.weight_hh_l0, m.bias_ih_l0, m.bias_hh_l0)


torch.manual_seed(SEED)
lstm = LSTMLM(V, D, D_H).to(DEVICE)

# nn.LSTM and your lstm_cell_step agree, step by step, on the untrained model.
with torch.no_grad():
    _x = torch.randint(0, V, (2, 6), device=DEVICE)
    _hs, _ = lstm.lstm(lstm.embed(_x))
    _h = _c = torch.zeros(2, D_H, device=DEVICE)
    for _t in range(6):
        _h, _c = lstm_cell_step(lstm.embed(_x)[:, _t], _h, _c, lstm.params())
        assert torch.allclose(_h, _hs[:, _t], atol=1e-5), "nn.LSTM differs from lstm_cell_step"

print(f"LSTM LM, {n_params(lstm):,} parameters ({n_params(rnn):,} for the RNN), {STEPS['LSTM']} steps")
started = time.time()
lstm_log = train_lm(lstm, STEPS["LSTM"], LR, KAPPA)
timings["LSTM training"] = time.time() - started
results["LSTM"] = {"val": evaluate_lm(lstm, "val")}
timings["LSTM evaluation (val)"] = time.time() - started - timings["LSTM training"]
print(f"LSTM val: {results['LSTM']['val']['nats']:.4f} nats/char, PPL {results['LSTM']['val']['ppl']:.3f}"
      f"   (training {timings['LSTM training']:.0f} s)")

**Run: the gradient–distance plot again.** The same measurement as in Exercise 3A, on the same 64 validation chunks, now for the trained LSTM. It is unrolled with *your* `lstm_cell_step` and the trained weights, so the hidden states can be inspected one step at a time.

**Predict.** Will the LSTM's curve lie above or below the RNN's? Will it still fall?

In [ ]:
started = time.time()
_p = lstm.params()
forget_gates = []  # f_t at every step, to see what the direct path multiplies by


def lstm_step(x_t, state):
    with torch.no_grad():
        forget_gates.append(torch.sigmoid(state[0] @ _p.W_f.T + x_t @ _p.U_f.T + _p.b_f))
    h, c = lstm_cell_step(x_t, *state, _p)
    return h, (h, c)


_zeros = torch.zeros(64, D_H, device=DEVICE)
lstm_dist = grad_norm_by_distance(lstm_step, (_zeros, _zeros), lstm.embed, lstm.out, dist_x, dist_y)
lstm.zero_grad()
timings["Gradient measurement (LSTM)"] = time.time() - started

plot_grad_distance([("RNN", rnn_dist, NAVY), ("LSTM", lstm_dist, ACCENT)])
print("distance      RNN        LSTM")
for d in (1, 10, 25, 50, 100, DIST_LEN - 1):
    print(f"   {d:>5}   {rnn_dist[d]:.2e}   {lstm_dist[d]:.2e}")
_f = torch.stack(forget_gates[1:])  # (L - 1, B, d_h); step 0 starts from h_0 = 0
print(f"\nForget gates f_t over these chunks: mean {_f.mean():.2f}; "
      f"{(_f > 0.9).float().mean():.0%} of entries above 0.9, "
      f"{(_f > 0.99).float().mean():.0%} above 0.99")

In [ ]:
# Checkpoint 4 follow-up: 50 characters back, the LSTM's gradient is far larger than the RNN's.
workshop.checkpoint(4, label="4 (follow-up)")
LSTM_OVER_RNN_MIN = 1e3  # measured at distance 50: 5e4 at full settings, 2e6 and 7e7 with QUICK
_ratio = (lstm_dist[50] / rnn_dist[50]).item() if rnn_dist[50] > 0 else math.inf
print(f"At distance 50 the LSTM's gradient is {_ratio:.1e} times the RNN's")
assert _ratio > LSTM_OVER_RNN_MIN, "the LSTM's gradient should reach much further back than the RNN's"
print("Checkpoint passed")

**Explain.** Both curves fall, but the LSTM's falls far more slowly, so a character 50 or 100 positions back still contributes a measurable gradient. That is the direct path $\partial c_t / \partial c_s = \prod_j \mathrm{diag}(f_j)$ at work: where the learned forget gates stay close to 1, nothing on that path shrinks the gradient. The LSTM does not *guarantee* long memory; it makes it learnable. Its curve still falls, because the forget gates are below 1 on average and the other paths through the gates do decay.

In [ ]:
# The results table: every model scored on exactly the same 60,394 test characters, once.
started = time.time()
for name, model in [("RNN", rnn), ("LSTM", lstm)]:
    results[name]["test"] = evaluate_lm(model, "test")
timings["Test evaluation (RNN, LSTM)"] = time.time() - started

print(f"Tiny Shakespeare, character level, test split ({len(ids['test']):,} characters), |V| = {V}\n")
print(f" {'model':<8} {'val nats':>9} {'test nats/char':>15} {'test PPL':>9} {'test BPC':>9}")
for name in ["2-gram", "3-gram", "5-gram", "RNN", "LSTM"]:
    v, t = results[name]["val"], results[name]["test"]
    print(f" {name:<8} {v['nats']:>9.4f} {t['nats']:>15.4f} {t['ppl']:>9.3f} {t['bpc']:>9.4f}")
print(f"\n Uniform over |V|: {math.log(V):.4f} nats, PPL {V}")

In [ ]:
# Checkpoint: the LSTM beats the character trigram on test, by a margin set from measured runs.
# Measured gap to the trigram: 3.04 PPL at full settings (seed 0), 2.15 with QUICK = True (seeds 1, 2).
workshop.checkpoint(label="LSTM against trigram")
PPL_MARGIN = 1.0
lstm_ppl, trigram_ppl = results["LSTM"]["test"]["ppl"], results["3-gram"]["test"]["ppl"]
fivegram_ppl = results["5-gram"]["test"]["ppl"]
print(f"LSTM test PPL {lstm_ppl:.3f}: {trigram_ppl - lstm_ppl:.3f} below the trigram ({trigram_ppl:.3f}), "
      f"{fivegram_ppl - lstm_ppl:.3f} below Lab 1's best, the 5-gram ({fivegram_ppl:.3f})")
assert lstm_ppl < trigram_ppl - PPL_MARGIN, "the LSTM should beat the character trigram clearly"
assert lstm_ppl < fivegram_ppl, "the LSTM should also beat Lab 1's best character n-gram"
print("Checkpoint passed: the LSTM beats the n-gram baseline at the same token unit and test text")

**Explain.** Every number in the table is a character-level cross-entropy on the same 60,394 test characters, over the same 65-symbol vocabulary, computed by the same function, with every setting chosen on val. So the differences are real. Each extra character of n-gram context helps less, because the counts get sparse. The neural models condition on the whole prefix through a learned state and share statistical strength between similar contexts, which add-$k$ counts cannot do. The gap between the RNN and the LSTM is consistent with the gradient plot, since the LSTM can learn from dependencies the RNN's gradient never reaches. But the two were not trained at equal compute (the LSTM had twice the steps and about 3.4 times the parameters), so this table alone does not show how much of the gap is due to gating.

# Part C · Generating Text

## Exercise 5 · Sampling With Temperature (8 minutes)

To generate, pick the next character from $\hat{y}_t$, feed it back in, repeat. **Temperature** $\tau > 0$ divides the logits before the softmax (@eq-temp):

$$
\hat{y}_t^{(\tau)} = \mathrm{softmax}(z_t / \tau), \qquad w_{t+1} \sim \hat{y}_t^{(\tau)}.
$$

**Predict.** For the logits $z = (2, 1, 0.5, 0, -1)$, which outcome is most likely at $\tau = 0.5$, $1$ and $2$, and does its probability go up or down as $\tau$ grows? What happens at $\tau = 0.01$?

**Task.** Complete `sample_with_temperature(logits, tau, generator)`. `logits` has shape `(..., |V|)`; return one sampled ID per row, shape `(...)`. Pass `generator` to `torch.multinomial`, so that a seed reproduces the draws. `torch.multinomial` accepts a 1-D or 2-D tensor of probabilities.

<details>
<summary>Hint for TODO 5</summary>

Divide the logits by $\tau$ before the softmax (briefing @eq-temp): multiplying makes $\tau = 0.5$ flatter instead of sharper, and the checkpoint's frequencies at $\tau = 0.5$ and $2$ will show it. `torch.multinomial` returns one column per draw, shape `(N, 1)`, so drop that axis and restore the leading shape: a 1-D input must come back as a single ID, a tensor of shape `()`.

</details>

In [ ]:
# TODO 5: complete this function.
def sample_with_temperature(logits, tau, generator=None):
    """One draw per row of `logits` (..., |V|) from softmax(logits / tau). Returns IDs, shape (...)."""
    raise NotImplementedError("TODO 5")

In [ ]:
#@title Solution 5 — try it yourself first { display-mode: "form" }
@workshop.solution(5)
def sample_with_temperature(logits, tau, generator=None):
    """One draw per row of `logits` (..., |V|) from softmax(logits / tau). Returns IDs, shape (...)."""
    probs = torch.softmax(logits / tau, dim=-1)  # ŷ^(τ), (..., |V|)
    flat = probs.reshape(-1, probs.shape[-1])  # (N, |V|): multinomial takes 1-D or 2-D input
    draws = torch.multinomial(flat, num_samples=1, generator=generator)  # (N, 1)
    return draws.reshape(probs.shape[:-1])  # (...)

<details>
<summary>Why this solution works</summary>

Dividing by $\tau < 1$ stretches the gaps between logits, so the softmax concentrates on the largest; $\tau > 1$ shrinks the gaps and flattens the distribution toward uniform. At $\tau = 0.01$ the gaps are multiplied by 100 and essentially all the mass sits on the arg max, so sampling becomes greedy decoding. Temperature never changes the *order* of the probabilities, only how sharp they are, and it leaves the model and its perplexity untouched.

</details>

In [ ]:
# Checkpoint 5: empirical frequencies over 20,000 draws match softmax(z / tau).
workshop.checkpoint(5)
TV_TOL = 0.02  # total variation; the largest measured over 200 seeds and 3 temperatures was 0.012
_z = torch.tensor([2.0, 1.0, 0.5, 0.0, -1.0])
_g = torch.Generator().manual_seed(5)
for _tau in (0.5, 1.0, 2.0):
    _draws = sample_with_temperature(_z.expand(20_000, -1), _tau, _g)
    assert _draws is not None and _draws.shape == (20_000,), (
        f"one ID per row: expected shape (20000,), got {None if _draws is None else tuple(_draws.shape)}: "
        "drop the draw axis that torch.multinomial adds")
    _freq = torch.bincount(_draws, minlength=5).float() / 20_000
    _tv = 0.5 * (_freq - torch.softmax(_z / _tau, dim=-1)).abs().sum().item()
    print(f"tau = {_tau}: empirical {[round(f, 3) for f in _freq.tolist()]}  total variation {_tv:.4f}")
    assert _tv < TV_TOL, (
        f"frequencies at tau = {_tau} do not match softmax(z / tau): total variation {_tv:.4f} >= {TV_TOL}; "
        "divide the logits by tau before the softmax, do not multiply")
_draws = sample_with_temperature(_z.expand(20_000, -1), 0.01, _g)
assert (_draws == 0).all(), "at tau = 0.01 every draw must be the arg max"
_a = sample_with_temperature(_z, 1.0, torch.Generator().manual_seed(7))
_b = sample_with_temperature(_z, 1.0, torch.Generator().manual_seed(7))
assert _a.shape == () and _a == _b, (
    f"a 1-D input gives one ID, a tensor of shape (), and the same seed the same ID; "
    f"got shape {tuple(_a.shape)}, values {_a.tolist()} and {_b.tolist()}")
print("Checkpoint 5 passed")

In [ ]:
@torch.no_grad()
def generate(model, prompt, length, pick):
    """Read `prompt`, then append `length` characters, each chosen by pick(next-character logits)."""
    model.eval()
    x = torch.tensor([[char_to_id[ch] for ch in prompt]], device=DEVICE)
    logits, state = model(x)
    out = []
    for _ in range(length):
        next_id = int(pick(logits[0, -1].float().cpu()))  # logits for the next character, (|V|,)
        out.append(chars[next_id])
        logits, state = model(torch.tensor([[next_id]], device=DEVICE), state)  # feed it back in
    model.train()
    return prompt + "".join(out)


started = time.time()
gen = torch.Generator().manual_seed(SEED)
prompt = "ROMEO:\n"
print("--- LSTM, greedy (arg max at every step) ---")
print(generate(lstm, prompt, 300, lambda z: z.argmax()))
for tau in (0.5, 1.0, 1.5):
    print(f"\n--- LSTM, tau = {tau} ---")
    print(generate(lstm, prompt, 300, lambda z, tau=tau: sample_with_temperature(z, tau, gen)))
print("\n--- RNN, tau = 1.0, for comparison ---")
print(generate(rnn, prompt, 300, lambda z: sample_with_temperature(z, 1.0, gen)))
timings["Generation"] = time.time() - started

**Explain.** Write two sentences in the cell below (double-click to edit): how do the samples change from greedy to $\tau = 1.5$, and which setting would you use?

*Your answer:*

Things to notice. Greedy decoding soon repeats itself: the arg max of each step leads back into the same few high-probability phrases. Low temperature gives safe, repetitive text; $\tau = 1$ samples the model's own distribution, with more variety and more invented words; at $\tau = 1.5$ the tail of unlikely characters gets enough mass that spelling breaks down. Compare with Lab 1's n-gram samples: the LSTM keeps the layout of a play (speaker names, line breaks) over many lines, which no 5-character window can. The stretch section adds two ways to cut the unlikely tail without flattening the whole distribution.

## The Results Card

Every number below was measured in this run. **Lab 5** compares its mini-GPT with this LSTM using exactly this metric: character-level cross-entropy in **nats per character** on the 60,394 characters of the **test** split (with perplexity $= e^{\text{nats}}$ and bits per character $=$ nats$/\ln 2$ beside it), over the 65 characters of train, computed by `lm_loss_and_ppl`, with every setting chosen on val. Each character is scored once, from the true text before it.

The cell also writes the card to `lab03_results.json` in the working directory, including the gradient-norm curves and the clipping traces. The values recorded in the repository (`data/baselines.json`) were produced by this cell.

In [ ]:
settings = {
    "d": D, "d_h": D_H, "layers": 1, "chunk_T": 128, "batch_size": 64, "optimizer": "Adam",
    "lr": LR, "kappa": KAPPA, "steps": STEPS, "quick": QUICK, "seed": SEED, "device": DEVICE,
    "evaluation": "every character of the split; state carried across 1,000-character chunks; "
                  "warm-up on the 1,000 characters before the split, not scored",
}
card = {
    "metric": "character-level cross-entropy, nats per character, Tiny Shakespeare test split",
    "results": results,
    "ngram_k": ngram_k,
    "settings": settings,
    "gradient_norm_by_distance": {"RNN": rnn_dist.tolist(), "LSTM": lstm_dist.tolist(),
                                  "spectral_norm_W_rnn": spectral_W},
    "clipping_demo": clip_demo,
    "training_curves": {"RNN": rnn_log, "LSTM": lstm_log},
    "timings_seconds": {k: round(v, 1) for k, v in timings.items()},
    "versions": {"torch": torch.__version__, "numpy": np.__version__},
}
with open("lab03_results.json", "w", encoding="utf-8") as f:
    json.dump(card, f)

print("RESULTS CARD · Lab 3 · measured in this run" + ("  (QUICK: not the baselines)" if QUICK else ""))
print("=" * 70)
print(f"Character LM, Tiny Shakespeare, test split ({len(ids['test']):,} characters)")
print(f" {'model':<8} {'nats/char':>10} {'perplexity':>11} {'bits/char':>10}   settings")
for name in ["3-gram", "5-gram", "RNN", "LSTM"]:
    t = results[name]["test"]
    extra = f"k = {ngram_k[int(name[0])]}" if "gram" in name else f"{STEPS[name]} steps, d_h = {D_H}"
    print(f" {name:<8} {t['nats']:>10.4f} {t['ppl']:>11.4f} {t['bpc']:>10.4f}   {extra}")
print("=" * 70)
print("Time per section (seconds):")
for section, seconds in timings.items():
    print(f"  {section:<32} {seconds:>6.1f}")
print(f"Run time so far: {time.time() - LAB_START:.0f} s. This is the end of the core path.")

## Stretch (optional) · Top-$k$ and Nucleus Sampling

For those who finish early. Nothing later in the workshop depends on this code; Modules 7 and 8 use the same ideas through library settings.

Temperature reshapes the whole distribution. Two filters instead cut off the unlikely tail and leave the rest as it is. Each returns logits with the excluded characters set to $-\infty$, so its output can go straight into your `sample_with_temperature` (whose softmax gives them probability 0).

- **Top-$k$:** keep the $k$ largest logits.
- **Nucleus (top-$p$)** (Holtzman et al., 2020): sort the probabilities in decreasing order and keep the smallest prefix whose total is at least $p$. When the model is confident the prefix is short; when it is unsure, it is long.

**Predict.** For the probabilities $(0.5, 0.25, 0.15, 0.1)$, which tokens does nucleus sampling keep at $p = 0.6$? At $p = 0.4$?

**Task.** Complete `top_k_filter(logits, k)` and `nucleus_filter(logits, p)` for logits of shape `(..., |V|)`. Useful pieces: `torch.topk`, `Tensor.sort`, `Tensor.cumsum`, `Tensor.scatter`, `Tensor.masked_fill`.

In [ ]:
# TODO 6 (stretch): complete both functions.
def top_k_filter(logits, k):
    """Keep the k largest logits in each row; set the others to -inf."""
    raise NotImplementedError("TODO 6")


def nucleus_filter(logits, p):
    """Keep the smallest set of most probable tokens with total probability >= p; others -inf."""
    raise NotImplementedError("TODO 6")

In [ ]:
#@title Solution 6 — try it yourself first { display-mode: "form" }
@workshop.solution(6)
def top_k_filter(logits, k):
    """Keep the k largest logits in each row; set the others to -inf."""
    top = torch.topk(logits, k, dim=-1).indices  # (..., k)
    keep = torch.zeros_like(logits, dtype=torch.bool).scatter(-1, top, True)  # exactly k per row
    return logits.masked_fill(~keep, float("-inf"))


@workshop.solution(6)
def nucleus_filter(logits, p):
    """Keep the smallest set of most probable tokens with total probability >= p; others -inf."""
    probs_desc, order = torch.softmax(logits, dim=-1).sort(dim=-1, descending=True)
    mass_before = probs_desc.cumsum(dim=-1) - probs_desc  # probability of the tokens ranked above
    drop_desc = mass_before >= p  # the tokens above already reach p, so this one is not needed
    drop_desc[..., 0] = False  # the most probable token is always kept
    drop = torch.zeros_like(drop_desc).scatter(-1, order, drop_desc)  # back to vocabulary order
    return logits.masked_fill(drop, float("-inf"))

<details>
<summary>Why this solution works</summary>

**Top-$k$.** `topk` finds the indices of the $k$ largest logits; `scatter` turns them into a mask with exactly $k$ entries per row, even when logits tie.

**Nucleus.** After sorting, a token belongs to the nucleus exactly when the tokens ranked above it have total probability *below* $p$: if they already reach $p$, the smallest prefix stops before it. `mass_before` is that total (the running sum minus the token's own probability). When the top token alone exceeds $p$, every other token has `mass_before` $\ge p$ and only the top token survives. With $p = 1$ nothing is dropped in exact arithmetic; in floating point the running sum can reach 1 a little early, so a token with a vanishingly small probability may be dropped, which changes nothing a sampler would produce.

</details>

In [ ]:
# Checkpoint 6 (stretch)
workshop.checkpoint(6)
torch.manual_seed(6)
_z = torch.randn(3, V)
for _k in (1, 5, V):
    _f = top_k_filter(_z, _k)
    assert (torch.isfinite(_f).sum(dim=-1) == _k).all(), f"top-k must keep exactly {_k} per row"
    assert torch.equal(_f[torch.isfinite(_f)], _z[torch.isfinite(_f)]), "kept logits must not change"
assert torch.equal(top_k_filter(_z, V), _z), "k = |V| must leave the logits unchanged"
assert (top_k_filter(_z, 1).argmax(-1) == _z.argmax(-1)).all()

_logits = torch.log(torch.tensor([0.15, 0.5, 0.1, 0.25]))  # sorted: 0.5, 0.25, 0.15, 0.1
for _p, _kept in [(0.6, [1, 3]), (0.4, [1]), (0.85, [0, 1, 3]), (0.95, [0, 1, 2, 3])]:
    _got = torch.isfinite(nucleus_filter(_logits, _p)).nonzero().flatten().tolist()
    assert _got == _kept, f"p = {_p}: kept {_got}, expected {_kept}"
assert torch.equal(nucleus_filter(_logits, 1.0), _logits), "p = 1 must keep every token"
assert torch.equal(nucleus_filter(_z, 1.0), _z), "p = 1 must leave the logits unchanged"
assert (torch.isfinite(nucleus_filter(_z, 1e-6)).sum(-1) == 1).all(), "tiny p keeps only the top token"
print("Checkpoint 6 passed")

In [ ]:
gen = torch.Generator().manual_seed(SEED)
for label, pick in [
    ("top-k, k = 10", lambda z: sample_with_temperature(top_k_filter(z, 10), 1.0, gen)),
    ("nucleus, p = 0.9", lambda z: sample_with_temperature(nucleus_filter(z, 0.9), 1.0, gen)),
    ("plain, tau = 1.0", lambda z: sample_with_temperature(z, 1.0, gen)),
]:
    print(f"--- LSTM, {label} ---\n{generate(lstm, prompt, 300, pick)}\n")

# How large is the nucleus? It changes from step to step, unlike k.
with torch.no_grad():
    _z, _ = lstm(ids["val"][None, :2000].to(DEVICE))
    _sizes = torch.isfinite(nucleus_filter(_z[0].float().cpu(), 0.9)).sum(dim=-1)
print(f"Nucleus size at p = 0.9 over 2,000 validation positions: median {_sizes.median().item()}, "
      f"min {_sizes.min().item()}, max {_sizes.max().item()} of {V}")

**Explain.** Both filters remove the tail of unlikely characters that produces misspellings at $\tau = 1$, and the samples become cleaner without the repetition of greedy decoding. The last line shows why nucleus sampling adapts: after "the" followed by a space, many characters are plausible and the nucleus is large; in the middle of a word one or two characters carry almost all the mass and the nucleus shrinks to them. A fixed $k$ cannot do both.

In [ ]:
# Generated by scripts/gen_notebooks.py: do not edit this cell.
# Which checkpoints passed, and on whose code; then a run record for the instructor.
workshop.summary()
workshop.run_record()

<!-- GENERATED by scripts/gen_notebooks.py. Do not edit this cell. -->
---
**Next:** [4 · Seq2seq and Attention](https://colab.research.google.com/github/project-delphi/nlp-llms/blob/main/notebooks/04-seq2seq-attention.ipynb)

[Workshop site](https://project-delphi.github.io/nlp-llms) · [All notebooks](https://project-delphi.github.io/nlp-llms/notebooks.html) · [Source](https://github.com/project-delphi/nlp-llms)

Genial Labs · text CC BY 4.0, code MIT